# **Part - A**

In [15]:
import cv2
import numpy as np
from PIL import Image, ImageSequence
from __future__ import annotations
import argparse
import pandas as pd
from pathlib import Path
import sys
import shutil
import os

def ExtractFrames(gifPath="input.gif", outDirAll="frames_all", outDirBad="frames_corrupted", badIdx=[0, 4, 9, 14, 18, 23, 28, 32, 37, 42]):
    os.makedirs(outDirAll, exist_ok=True)
    os.makedirs("frames_restored", exist_ok=True)
    cap = cv2.VideoCapture(gifPath)
    if not cap.isOpened():
        raise FileNotFoundError(f"Could not open {gifPath}")
    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
        cv2.imwrite(os.path.join(outDirAll, f"{idx:03d}.png"), frame)
        idx += 1
    cap.release()
    if os.path.exists(outDirBad):
        shutil.rmtree(outDirBad)
    os.makedirs(outDirBad, exist_ok=True)
    copied = []
    for i in badIdx:
        src = os.path.join(outDirAll, f"{i:03d}.png")
        if os.path.exists(src):
            dst = os.path.join(outDirBad, f"{i:03d}.png")
            shutil.copyfile(src, dst)
            copied.append(i)
    print(f"Extracted {len(frames)} total frames to ./{outDirAll}")
    print(f"Copied corrupted frames {copied} to ./{outDirBad}\n\n")

# **Part - B**

In [16]:
#Helper functions for image processing of IMG 00.png
def BlockinessStats(Img, B):
    G = cv2.cvtColor(Img, cv2.COLOR_BGR2GRAY)
    H, W = G.shape
    Dv = np.abs(G[:, 1:] - G[:, :-1]); Mv = (np.arange(1, W) % B == 0)
    Dh = np.abs(G[1:, :] - G[:-1, :]); Mh = (np.arange(1, H) % B == 0)
    Vb, Vn = Dv[:, Mv].mean(), Dv[:, ~Mv].mean()
    Hb, Hn = Dh[Mh, :].mean(), Dh[~Mh, :].mean()
    Nb = (Vn + Hn) / 2.0
    return (Vb / Vn, Hb / Hn), Nb

def BoundaryMask(H, W, B, HalfWidth=2):
    M = np.zeros((H, W), np.uint8)
    for X in range(B, W, B):
        X0, X1 = max(0, X - HalfWidth), min(W, X + HalfWidth)
        M[:, X0:X1] = 255
    for Y in range(B, H, B):
        Y0, Y1 = max(0, Y - HalfWidth), min(H, Y + HalfWidth)
        M[Y0:Y1, :] = 255
    return M

def FeatherAlpha(BMask, EdgeBand=2, FeatherMult=3.0, BlurSigma=0.8):
    Inv = (BMask == 0).astype(np.uint8) * 255
    Dist = cv2.distanceTransform(Inv, cv2.DIST_L2, 3)
    A = np.clip(1.0 - (Dist / (FeatherMult * EdgeBand + 1e-6)), 0.0, 1.0)
    return cv2.GaussianBlur(A, (0, 0), BlurSigma).astype(np.float32)

def BilateralAny(Img, D, SigmaC, SigmaS):
    X = Img if Img.dtype == np.uint8 else np.clip(Img, 0, 255).astype(np.uint8)
    if X.ndim == 2 or (X.ndim == 3 and X.shape[2] in (1, 3)):
        return cv2.bilateralFilter(X, d=D, sigmaColor=SigmaC, sigmaSpace=SigmaS)
    Ch = cv2.split(X)
    Ch = [cv2.bilateralFilter(C, d=D, sigmaColor=SigmaC, sigmaSpace=SigmaS) for C in Ch]
    return cv2.merge(Ch)

def SoftDeblock(Img, B, EdgeBand=3, BilD=9, SigC=30, SigS=7):
    H, W = Img.shape[:2]
    BMask = BoundaryMask(H, W, B, HalfWidth=EdgeBand)
    Alpha = FeatherAlpha(BMask, EdgeBand=EdgeBand)
    Smooth = BilateralAny(Img, D=BilD, SigmaC=SigC, SigmaS=SigS).astype(np.float32)
    Base = Img.astype(np.float32)
    if Img.ndim == 2:
        Out = Alpha * Smooth + (1.0 - Alpha) * Base
    else:
        Out = Alpha[..., None] * Smooth + (1.0 - Alpha[..., None]) * Base
    return np.clip(Out, 0, 255).astype(np.uint8)


#Helper functions for image processing of IMG 04.png
def Rotate_4(img, deg, borderMode, borderValue=(0,0,0)):
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), -deg, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=borderMode, borderValue=borderValue)

def EccAffineAlign_4(targetGray, srcBgr, iters=300, eps=1e-6):
    h, w = targetGray.shape[:2]
    W = np.eye(2,3, dtype=np.float32)
    sc = 0.5
    tg = cv2.resize(targetGray, (int(w*sc), int(h*sc)))
    sg = cv2.resize(cv2.cvtColor(srcBgr, cv2.COLOR_BGR2GRAY), (int(w*sc), int(h*sc)))
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, iters, eps)
    try:
        _, Ws = cv2.findTransformECC(tg, sg, W, cv2.MOTION_AFFINE, crit)
        Ws[:,2] /= sc
        W = Ws
    except cv2.error:
        pass
    return cv2.warpAffine(srcBgr, W, (w,h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT_101)

def OneSidedFeather(maskU8, featherPx=32):
    if featherPx <= 0:
        return (maskU8>0).astype(np.float32)[...,None]
    dist = cv2.distanceTransform(maskU8, cv2.DIST_L2, 3)
    return np.clip(dist/float(featherPx), 0, 1).astype(np.float32)[...,None]

def TranslateXY(img, tx, ty, borderMode=cv2.BORDER_REFLECT_101):
    M = np.float32([[1,0,tx],[0,1,ty]])
    h, w = img.shape[:2]
    return cv2.warpAffine(img, M, (w,h), flags=cv2.INTER_CUBIC, borderMode=borderMode)


#Helper functions for image processing of IMG 32.png
def ImreadRgbFloat(Path):
    Bgr = cv2.imread(str(Path), cv2.IMREAD_COLOR)
    if Bgr is None:
        raise RuntimeError(f"OpenCV failed to read: {Path}")
    Rgb = cv2.cvtColor(Bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return Rgb

def ToUint8(Rgb):
    return (np.clip(Rgb, 0.0, 1.0) * 255.0 + 0.5).astype(np.uint8)

def MakeGaussianPsf(Size, Sigma):
    Ax = np.arange(-(Size // 2), (Size // 2) + 1, dtype=np.float32)
    Xx, Yy = np.meshgrid(Ax, Ax)
    Psf = np.exp(-(Xx**2 + Yy**2) / (2 * Sigma**2))
    Psf /= Psf.sum()
    return Psf

def WienerDeconvChannel(Channel, Psf, K):
    H, W = Channel.shape
    PsfPad = np.zeros((H, W), dtype=np.float32)
    Ph, Pw = Psf.shape
    Y0, X0 = (H - Ph) // 2, (W - Pw) // 2
    PsfPad[Y0:Y0+Ph, X0:X0+Pw] = Psf
    PsfShift = np.fft.ifftshift(PsfPad)
    G = np.fft.fft2(Channel)
    Hf = np.fft.fft2(PsfShift)
    HConj = np.conj(Hf)
    Denom = np.abs(Hf)**2 + K
    FHat = (HConj / Denom) * G
    FEst = np.fft.ifft2(FHat).real
    FEst = np.clip(FEst, 0.0, 1.0)
    return FEst.astype(np.float32)

def VarianceOfLaplacian(Rgb):
    Gray = cv2.cvtColor(ToUint8(Rgb), cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(Gray, cv2.CV_64F).var()

def UnsharpMask(Rgb, RadiusSigma=1.0, Amount=0.4, Threshold=0.0):
    Blurred = cv2.GaussianBlur(Rgb, (0, 0), sigmaX=RadiusSigma, sigmaY=RadiusSigma)
    Detail = Rgb - Blurred
    if Threshold > 0:
        Mask = (np.abs(Detail) >= Threshold).astype(np.float32)
        Detail *= Mask
    Out = np.clip(Rgb + Amount * Detail, 0.0, 1.0)
    return Out

def ReplaceBordersWithRef(img, ref, left, right, top, bottom):
    H, W, _ = img.shape
    out = img.copy()
    left   = int(max(0, min(left,   W)))
    right  = int(max(0, min(right,  W)))
    top    = int(max(0, min(top,    H)))
    bottom = int(max(0, min(bottom, H)))

    if left > 0:
        out[:, :left, :] = ref[:, :left, :]
    if right > 0:
        out[:, W-right:, :] = ref[:, W-right:, :]
    if top > 0:
        out[:top, :, :] = ref[:top, :, :]
    if bottom > 0:
        out[H-bottom:, :, :] = ref[H-bottom:, :, :]
    return out


#Helper functions for image processing of IMG 37.png
def Rotate_37(Img, Deg, BorderMode, BorderValue=(0,0,0)):
    H, W = Img.shape[:2]
    M = cv2.getRotationMatrix2D((W/2, H/2), -Deg, 1.0)
    return cv2.warpAffine(Img, M, (W, H), flags=cv2.INTER_CUBIC,
                          borderMode=BorderMode, borderValue=BorderValue)

def EccAffineAlign_37(TargetGray, SrcBgr, Iters=300, Eps=1e-6):
    H, W = TargetGray.shape[:2]
    Wm = np.eye(2,3, dtype=np.float32); Sc = 0.5
    Tg = cv2.resize(TargetGray, (int(W*Sc), int(H*Sc)))
    Sg = cv2.resize(cv2.cvtColor(SrcBgr, cv2.COLOR_BGR2GRAY), (int(W*Sc), int(H*Sc)))
    Crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, Iters, Eps)
    try:
        _, Ws = cv2.findTransformECC(Tg, Sg, Wm, cv2.MOTION_AFFINE, Crit)
        Ws[:,2] /= Sc; Wm = Ws
    except cv2.error:
        pass
    return cv2.warpAffine(SrcBgr, Wm, (W, H), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_REFLECT_101)

def LabMatchBorder(DonorBgr, TargetBgr, MaskU8, RingPx=15):
    Ring = cv2.subtract(MaskU8, cv2.erode(MaskU8, np.ones((RingPx, RingPx), np.uint8), 1))
    if Ring.max() == 0:
        return DonorBgr
    R = Ring.astype(bool)
    Dlab = cv2.cvtColor(DonorBgr,  cv2.COLOR_BGR2LAB).astype(np.float32)
    Tlab = cv2.cvtColor(TargetBgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    for ch in range(3):
        D = Dlab[..., ch][R]; T = Tlab[..., ch][R]
        if D.size < 50:
            continue
        Md, Sd = float(np.mean(D)), float(np.std(D) + 1e-6)
        Mt, St = float(np.mean(T)), float(np.std(T) + 1e-6)
        Dlab[..., ch] = (Dlab[..., ch] - Md) * (St / Sd) + Mt
    Dlab = np.clip(Dlab, 0, 255).astype(np.uint8)
    return cv2.cvtColor(Dlab, cv2.COLOR_LAB2BGR)


#Main processing function to restore all corrupted images and save the restored images to frames_restored folder
def Restoring_ALL():
    def Restore_0(InPath, OutPath):
        Img = cv2.imread(InPath, cv2.IMREAD_COLOR)
        B = 32
        (Vr0, Hr0), Nb0 = BlockinessStats(Img, B)
        Out = SoftDeblock(Img, B, EdgeBand=3, BilD=9, SigC=30, SigS=7)
        (Vr1, Hr1), Nb1 = BlockinessStats(Out, B)
        Blk = 0.5 * (Vr1 + Hr1)
        Drop = max(0.0, (Nb0 - Nb1) / max(Nb0, 1e-6))
        Tau, Lam = 0.10, 2.0
        Cost = (Blk - 1.0) ** 2 + Lam * max(0.0, Drop - Tau) ** 2
        cv2.imwrite(OutPath, Out)
        print("Saved -> 0.png")
    Restore_0(r"frames_corrupted\000.png", r"frames_restored\0.png")
    
    
    def Restore_4(F4, F5, DxTr, DyTr, DxBr, DyBr,DarkThr, BrightThr, FeatherDx, ExpandPx):
        f4 = cv2.imread(F4); f5 = cv2.imread(F5)
        h, w = f4.shape[:2]
        Base = Rotate_4(f4, 135, cv2.BORDER_REFLECT_101)
        RotBlack = Rotate_4(f4, 135, cv2.BORDER_CONSTANT, (0,0,0))
        F5Aligned = EccAffineAlign_4(cv2.cvtColor(Base, cv2.COLOR_BGR2GRAY), f5)
        Bw = (cv2.cvtColor(RotBlack, cv2.COLOR_BGR2GRAY) < 5).astype(np.uint8)*255
        Bw = cv2.morphologyEx(Bw, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
        Bw = cv2.dilate(Bw, np.ones((3,3), np.uint8), 1)
        yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
        Left = (xx < w//2); Right = ~Left
        Top = (yy < h//2); Bottom = ~Top
        Tl = (Bw>0) & Left & Top
        Tr = (Bw>0) & Right & Top
        Bl = (Bw>0) & Left & Bottom
        Br = (Bw>0) & Right & Bottom
        MaskLeftU8 = np.zeros_like(Bw, np.uint8); MaskLeftU8[(Tl|Bl)] = 255
        MaskTrU8 = np.zeros_like(Bw, np.uint8); MaskTrU8[Tr] = 255
        MaskBrU8 = np.zeros_like(Bw, np.uint8); MaskBrU8[Br] = 255
        Kernel = np.ones((2,2), np.uint8)
        MaskLeftU8 = cv2.erode(MaskLeftU8, Kernel, 1)
        MaskTrU8 = cv2.erode(MaskTrU8, Kernel, 1)
        MaskBrU8 = cv2.erode(MaskBrU8, Kernel, 1)
        AlphaLeft = OneSidedFeather(MaskLeftU8, 32)
        AlphaTr = OneSidedFeather(MaskTrU8, 32)
        AlphaBr = OneSidedFeather(MaskBrU8, 32)
        F5Tr = TranslateXY(F5Aligned, DxTr, DyTr)
        F5Br = TranslateXY(F5Aligned, DxBr, DyBr)
        Out = Base.copy()
        Out = (AlphaLeft * F5Aligned + (1.0 - AlphaLeft) * Out).astype(np.uint8)
        Out = (AlphaTr   * F5Tr      + (1.0 - AlphaTr)   * Out).astype(np.uint8)
        Out = (AlphaBr   * F5Br      + (1.0 - AlphaBr)   * Out).astype(np.uint8)
        GBase = cv2.cvtColor(Base, cv2.COLOR_BGR2GRAY)
        GTr   = cv2.cvtColor(F5Tr,  cv2.COLOR_BGR2GRAY)
        GBr   = cv2.cvtColor(F5Br,  cv2.COLOR_BGR2GRAY)
        CandTr = (MaskTrU8>0) & (GTr < DarkThr) & (GBase > BrightThr)
        CandBr = (MaskBrU8>0) & (GBr < DarkThr) & (GBase > BrightThr)
        DarkenU8 = np.zeros_like(Bw)
        DarkenU8[CandTr] = 255
        DarkenU8[CandBr] = 255
        if ExpandPx > 0:
            DarkenU8 = cv2.dilate(DarkenU8, np.ones((ExpandPx,ExpandPx), np.uint8), 1)
        DarkAlpha = OneSidedFeather(DarkenU8, FeatherDx)
        Out = ((1.0 - DarkAlpha) * Out).astype(np.uint8)
        cv2.imwrite(r"frames_restored\4.png", Out)
        print("Saved -> 4.png")
    Restore_4(F4=r"frames_all\004.png", F5=r"frames_all\003.png", DxTr=70, DyTr=70, DxBr=55, DyBr=50, DarkThr=65, BrightThr=180, FeatherDx=21, ExpandPx=6)
    
    
    def Restore_9(ImgBGR, Sigma, K):
        def Deconv2D(Chan):
            Hh, Ww = Chan.shape
            Fy = np.fft.fftfreq(Hh).reshape(-1,1)
            Fx = np.fft.fftfreq(Ww).reshape(1,-1)
            H = np.exp(-0.5 * ((2*np.pi*Sigma)**2) * (Fx*Fx + Fy*Fy))
            G = np.fft.fft2(Chan.astype(np.float32))
            Fhat = (np.conj(H) / (H*H + K)) * G
            Out = np.fft.ifft2(Fhat).real
            return np.clip(Out, 0, 255).astype(np.uint8)
        if ImgBGR.ndim == 2:
            return Deconv2D(ImgBGR)
        Ch = cv2.split(ImgBGR)
        Ch = [Deconv2D(C) for C in Ch]
        Ch = cv2.merge(Ch)
        cv2.imwrite(r"frames_restored\9.png", Ch)
        print("Saved -> 9.png")
    Restore_9(ImgBGR = cv2.imread(r"frames_all\009.png", cv2.IMREAD_COLOR), Sigma=1.4, K=0.012)
    
    
    def Restore_14(ImgBgr, Beta, BilD, SigmaC, SigmaS, NlmH):
        Lab = cv2.cvtColor(ImgBgr, cv2.COLOR_BGR2LAB).astype(np.float32)
        L, A, B = cv2.split(Lab)
        Base = cv2.bilateralFilter(L.astype(np.uint8), d=BilD, sigmaColor=SigmaC, sigmaSpace=SigmaS).astype(np.float32)
        Detail = L - Base
        L2 = np.clip(L - Beta * Detail, 0, 255).astype(np.uint8)
        Out = cv2.cvtColor(cv2.merge([L2, A.astype(np.uint8), B.astype(np.uint8)]), cv2.COLOR_LAB2BGR)
        Out = cv2.fastNlMeansDenoisingColored(Out, None, NlmH, NlmH, 7, 21)
        cv2.imwrite(r"frames_restored\14.png", Out)
        print("Saved -> 14.png")
    Restore_14(ImgBgr = cv2.imread(r"frames_all\014.png", cv2.IMREAD_COLOR), Beta=0.25, BilD=5, SigmaC=65, SigmaS=3, NlmH=4)
    
    
    def Restore_18(ImgBgr):
        HL=16; HC=14.5; T=3; S=9; G=0.52
        Lab=cv2.cvtColor(ImgBgr, cv2.COLOR_BGR2LAB)
        L,A,B=cv2.split(Lab)
        Ld=cv2.fastNlMeansDenoising(L,None,float(HL),T,S)
        Ad=cv2.fastNlMeansDenoising(A,None,float(HC),T,S)
        Bd=cv2.fastNlMeansDenoising(B,None,float(HC),T,S)
        Out=cv2.cvtColor(cv2.merge([Ld,Ad,Bd]),cv2.COLOR_LAB2BGR)
        if G>0: Out=cv2.GaussianBlur(Out,(0,0),G)
        cv2.imwrite(r"frames_restored\18.png", Out)
        print("Saved -> 18.png")
    Restore_18(cv2.imread(r"frames_all\018.png", cv2.IMREAD_COLOR))
    
    
    def Restore_23(Bgr, Ksize, HiThr, DeltaThr):
        G = cv2.cvtColor(Bgr, cv2.COLOR_BGR2GRAY)
        Se = cv2.getStructuringElement(cv2.MORPH_RECT, (Ksize, Ksize))
        Gmin = cv2.erode(G, Se, iterations=1)
        SaltMask = ((G >= HiThr) & ((G - Gmin) >= DeltaThr)).astype(np.uint8) * 255
        SaltMask = cv2.dilate(SaltMask, np.ones((3,3), np.uint8), 1)
        B, Gch, R = cv2.split(Bgr)
        Bmin = cv2.erode(B, Se, 1)
        Gminc = cv2.erode(Gch, Se, 1)
        Rmin = cv2.erode(R, Se, 1)
        MinBgr = cv2.merge([Bmin, Gminc, Rmin])
        Out = np.where(SaltMask[...,None] > 0, MinBgr, Bgr)
        cv2.imwrite(r"frames_restored\23.png", Out)
        print("Saved -> 23.png")
    Restore_23(Bgr= cv2.imread("frames_all/023.png"), Ksize=2, HiThr=250, DeltaThr=50)
    
    
    def Restore_28(Bgr, Ksize, LoThr, DeltaThr):
        G = cv2.cvtColor(Bgr, cv2.COLOR_BGR2GRAY)
        Se = cv2.getStructuringElement(cv2.MORPH_RECT, (Ksize, Ksize))
        Gmax = cv2.dilate(G, Se, 1)
        Pepper = ((G <= LoThr) & ((Gmax - G) >= DeltaThr)).astype(np.uint8) * 255
        Pepper = cv2.dilate(Pepper, np.ones((3,3), np.uint8), 1)
        B, Gc, R = cv2.split(Bgr)
        Bmx = cv2.dilate(B,  Se, 1)
        Gmx = cv2.dilate(Gc, Se, 1)
        Rmx = cv2.dilate(R,  Se, 1)
        MaxBgr = cv2.merge([Bmx, Gmx, Rmx])
        Out = np.where(Pepper[...,None] > 0, MaxBgr, Bgr)
        cv2.imwrite(r"frames_restored\28.png", Out)
        print("Saved -> 28.png")
    Restore_28(Bgr = cv2.imread("frames_all/028.png"), Ksize=2, LoThr=0, DeltaThr=5)
    
    
    def Restore_32(InputPath, RefPath, FinalOut):
        SrcRgb = ImreadRgbFloat(InputPath)
        RefRgb = ImreadRgbFloat(RefPath)
        PsfSize, Sigma = 13, 2.0
        K = 0.02
        Psf = MakeGaussianPsf(PsfSize, Sigma)
        WienerRgb = np.empty_like(SrcRgb, dtype=np.float32)
        for C in range(3):
            WienerRgb[..., C] = WienerDeconvChannel(SrcRgb[..., C], Psf, K)
        TargetSize = (1000, 1000)
        RefRgb_rs    = cv2.resize(RefRgb,    TargetSize, interpolation=cv2.INTER_AREA)
        WienerRgb_rs = cv2.resize(WienerRgb, TargetSize, interpolation=cv2.INTER_AREA)
        TargetSharp = VarianceOfLaplacian(RefRgb_rs)
        RadiusSigma = 1.0
        Threshold   = 0.01
        AmtMin, AmtMax = 0.00, 1.00
        TolFrac = 0.05
        NSteps  = 12
        BestImg  = WienerRgb_rs.copy()
        BestAmt  = 0.0
        BestDiff = float('inf')
        Low, High = AmtMin, AmtMax
        for _ in range(NSteps):
            Mid = 0.5 * (Low + High)
            Candidate = UnsharpMask(WienerRgb_rs, RadiusSigma=RadiusSigma, Amount=Mid, Threshold=Threshold)
            Sharpness = VarianceOfLaplacian(Candidate)
            Diff = abs(Sharpness - TargetSharp)
            if Diff < BestDiff:
                BestDiff = Diff
                BestImg  = Candidate
                BestAmt  = Mid
            if Sharpness < TargetSharp * (1 - TolFrac):
                Low = Mid
            elif Sharpness > TargetSharp * (1 + TolFrac):
                High = Mid
            else:
                BestImg  = Candidate
                BestAmt  = Mid
                break
        BestImg_with_ref_borders = ReplaceBordersWithRef(img=BestImg, ref=RefRgb_rs, left=650, right=125, top=450, bottom=250)
        Image.fromarray(ToUint8(BestImg_with_ref_borders)).save(FinalOut)
        print("Saved -> 32.png")
    Restore_32(InputPath = r"frames_all/032.png", RefPath = r"frames_all/033.png", FinalOut = r"frames_restored/32.png")
    
    
    def Restore_37(F37, F36, Out):
        Img37 = cv2.imread(F37); assert Img37 is not None
        Rot37Reflect = Rotate_37(Img37, 60.0, cv2.BORDER_REFLECT_101)
        Rot37Black   = Rotate_37(Img37, 60.0, cv2.BORDER_CONSTANT, (0,0,0))
        Img36 = cv2.imread(F36); assert Img36 is not None
        Donor36 = EccAffineAlign_37(cv2.cvtColor(Rot37Reflect, cv2.COLOR_BGR2GRAY), Img36)
        M = (cv2.cvtColor(Rot37Black, cv2.COLOR_BGR2GRAY) < 8).astype(np.uint8)*255
        M = cv2.morphologyEx(M, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
        M = cv2.dilate(M, np.ones((3,3), np.uint8), 1)
        M = cv2.GaussianBlur(M, (0,0), 1.2)
        Donor36Matched = LabMatchBorder(Donor36, Rot37Reflect, M, RingPx=15)
        Alpha = (M.astype(np.float32)/255.0)[...,None]
        Filled = (Alpha*Donor36Matched + (1.0-Alpha)*Rot37Black).astype(np.uint8)
        cv2.imwrite(Out, Filled)
        print(f"Saved -> 37.png")
    Restore_37(F37 = r"frames_all/037.png", F36 = r"frames_all/036.png", Out = r"frames_restored/37.png")
    
        
    def Restore_42():
        PreviousFrame = cv2.imread(r"frames_all\008.png")
        MiddleFrame = cv2.imread(r"frames_all\042.png")
        NextFrame = cv2.imread(r"frames_all\010.png")
        Height, Width = MiddleFrame.shape[:2]
        PreviousFrame = cv2.resize(PreviousFrame, (Width, Height))
        NextFrame = cv2.resize(NextFrame, (Width, Height))
        AverageNeighbors = cv2.addWeighted(PreviousFrame, 0.5, NextFrame, 0.5, 0)
        GrayMiddle = cv2.cvtColor(MiddleFrame, cv2.COLOR_BGR2GRAY)
        SmoothedGray = cv2.medianBlur(GrayMiddle, 5)
        MotionMask = (cv2.absdiff(GrayMiddle, SmoothedGray) > 6).astype(np.uint8)
        MotionMask = cv2.dilate(MotionMask, np.ones((3,3), np.uint8), iterations=2)
        MotionMask3 = np.repeat(MotionMask[:, :, None], 3, axis=2)
        FusedFrame = np.where(MotionMask3 == 1, AverageNeighbors, MiddleFrame)
        DeblockedFrame = cv2.bilateralFilter(FusedFrame, 7, 50, 5)
        BlurredFrame = cv2.GaussianBlur(DeblockedFrame, (0,0), 1.0)
        RestoredFrame = cv2.addWeighted(DeblockedFrame, 1.3, BlurredFrame, -0.3, 0)
        cv2.imwrite(r"frames_restored\42.png", RestoredFrame)
        print("Saved -> 42.png")
    Restore_42()
    print("All Frames Restored Successfully And Saved In ./frames_restored\n")

# Function to create submission.csv from the restored frames that were saved in frames_restored folder
    FramePaths: dict[int, str] = {
        0:  r"frames_restored\0.png",   4:  r"frames_restored\4.png",
        9:  r"frames_restored\42.png",  14: r"frames_restored\23.png",
        18: r"frames_restored\18.png",  23: r"frames_restored\14.png",
        28: r"frames_restored\28.png",  32: r"frames_restored\32.png",
        37: r"frames_restored\37.png",  42: r"frames_restored\9.png",
    }

    def LoadImage(PathObj: Path):
        Img = cv2.imread(str(PathObj), cv2.IMREAD_UNCHANGED)
        if Img is None:
            print(f"Failed to read image: {PathObj}")
            return None
        if Img.ndim == 2:
            Img = cv2.cvtColor(Img, cv2.COLOR_GRAY2BGR)
        return Img

    def BuildSubmission(Images: dict[int, np.ndarray]) -> pd.DataFrame:
        if not Images:
            print("No images provided to build submission")
            return pd.DataFrame({"ID": []})
        Sizes = [(Im.shape[0], Im.shape[1]) for Im in Images.values()]
        MinH = min(h for h, _ in Sizes)
        MinW = min(w for _, w in Sizes)
        print(f"Cropping all frames (if needed) to common size {MinW}x{MinH}")
        Processed = {}
        for Idx, Im in Images.items():
            if Im.shape[2] == 4:
                Im = Im[:, :, :3]
            if Im.shape[0] != MinH or Im.shape[1] != MinW:
                Im = Im[:MinH, :MinW]
            Processed[Idx] = Im
        Sample = next(iter(Processed.values()))
        H, W, C = Sample.shape
        NumPixels = H * W * C
        Data = {"ID": np.arange(NumPixels)}
        for FrameIdx in [0, 4, 9, 14, 18, 23, 28, 32, 37, 42]:
            if FrameIdx in Processed:
                Data[f"f{FrameIdx}"] = Processed[FrameIdx].reshape(-1)
            else:
                print(f"Missing frame {FrameIdx}")
        return pd.DataFrame(Data)

    def Arguments(Argv: list[str] | None = None) -> int:
        Parser = argparse.ArgumentParser(description="Generate submission.csv from explicit frame paths")
        Parser.add_argument("--output", "-o", default="submission.csv", help="Output CSV file path")
        if Argv is None:
            Args, _ = Parser.parse_known_args()
        else:
            Args = Parser.parse_args(Argv)

        FoundImages: dict[int, np.ndarray] = {}
        for FrameIdx in [0, 4, 9, 14, 18, 23, 28, 32, 37, 42]:
            P = FramePaths.get(FrameIdx)
            if not P:
                print(f"Missing frame {FrameIdx}")
                continue
            Img = LoadImage(Path(P))
            if Img is None:
                continue
            FoundImages[FrameIdx] = Img
            print(f"Loaded frame {FrameIdx} from {Path(P).name} shape={Img.shape}")

        if not FoundImages:
            print("No required frames found. Exiting.")
            return 1

        Df = BuildSubmission(FoundImages)
        Df.to_csv(Args.output, index=False)
        print(f"Wrote {Args.output} with columns: {list(Df.columns)} and {len(Df)} rows\n\n")
    Arguments()

# **Part - C**

In [17]:
def CreateVideo_MSE(InputGif="input.gif", CsvPath="submission.csv", OutGif="restored.gif", DurationMs=40):
    RequiredFrames = [0, 4, 9, 14, 18, 23, 28, 32, 37, 42]
    FramePaths = {
        0:  r"frames_restored\0.png",   4:  r"frames_restored\4.png",
        9:  r"frames_restored\42.png",  14: r"frames_restored\23.png",
        18: r"frames_restored\18.png",  23: r"frames_restored\14.png",
        28: r"frames_restored\28.png",  32: r"frames_restored\32.png",
        37: r"frames_restored\37.png",  42: r"frames_restored\9.png",
    }
    _ = pd.read_csv(CsvPath)
    im = Image.open(InputGif)
    frames_rgb, durations = [], []
    for f in ImageSequence.Iterator(im):
        frames_rgb.append(f.convert("RGB"))
        durations.append(int(f.info.get("duration", DurationMs)))
    w0, h0 = frames_rgb[0].size
    restored_pils = {k: Image.fromarray(cv2.cvtColor(cv2.resize(cv2.imread(FramePaths[k]), (w0, h0)), cv2.COLOR_BGR2RGB)) for k in RequiredFrames}
    timeline = [restored_pils.get(i, frames_rgb[i]) for i in range(len(frames_rgb))]
    timeline[0].save(OutGif, save_all=True, append_images=timeline[1:], loop=0, duration=durations)
    print("Restored.gif has been saved successfully")
    def read_bgr(p): return cv2.imread(p, cv2.IMREAD_COLOR)
    def mse(a, b):
        h = min(a.shape[0], b.shape[0]); w = min(a.shape[1], b.shape[1])
        A = a[:h, :w].astype(np.float32); B = b[:h, :w].astype(np.float32)
        return float(np.mean((A - B) ** 2))
    FramedPaths = {
        0:  r"frames_restored\0.png",   4:  r"frames_restored\4.png",
        9:  r"frames_restored\9.png",   14: r"frames_restored\14.png",
        18: r"frames_restored\18.png",  23: r"frames_restored\23.png",
        28: r"frames_restored\28.png",  32: r"frames_restored\32.png",
        37: r"frames_restored\37.png",  42: r"frames_restored\42.png",
    }
    mse_list = [mse(read_bgr(FramedPaths[k]), read_bgr(fr"frames_corrupted\{k:03d}.png")) for k in RequiredFrames]
    for i in range(len(mse_list)):
        print(f"MSE between Corrupt and Restored Frame {RequiredFrames[i]:03d}.png: {mse_list[i]:.2f}")
    print("\n")
    return mse_list

# **Part - D**

In [18]:
def RunAll():
    path_of_corrupted_file = "input.gif"
    print("Part A: Extracting frames from input gif and copying corrupted frames to frames_corrupted folder")
    ExtractFrames(gifPath=path_of_corrupted_file, outDirAll="frames_all", outDirBad="frames_corrupted", badIdx=[0, 4, 9, 14, 18, 23, 28, 32, 37, 42])
    print("Part B: Restoring all corrupted frames and saving them to frames_restored folder and creating submission.csv")
    Restoring_ALL()
    print("Part C: Making restored gif and calculating MSE values between corrupted and restored frames")
    CreateVideo_MSE(InputGif=path_of_corrupted_file, CsvPath="submission.csv", OutGif="restored.gif", DurationMs=40)
    print("Part D: Completed all tasks successfully!")

RunAll()

Part A: Extracting frames from input gif and copying corrupted frames to frames_corrupted folder
Extracted 43 total frames to ./frames_all
Copied corrupted frames [0, 4, 9, 14, 18, 23, 28, 32, 37, 42] to ./frames_corrupted


Part B: Restoring all corrupted frames and saving them to frames_restored folder and creating submission.csv
Saved -> 0.png
Saved -> 4.png
Saved -> 9.png
Saved -> 14.png
Saved -> 18.png
Saved -> 23.png
Saved -> 28.png
Saved -> 32.png
Saved -> 37.png
Saved -> 42.png
All Frames Restored Successfully And Saved In ./frames_restored

Loaded frame 0 from 0.png shape=(1000, 1000, 3)
Loaded frame 4 from 4.png shape=(1000, 1000, 3)
Loaded frame 9 from 42.png shape=(1000, 1000, 3)
Loaded frame 14 from 23.png shape=(1000, 1000, 3)
Loaded frame 18 from 18.png shape=(1000, 1000, 3)
Loaded frame 23 from 14.png shape=(1000, 1000, 3)
Loaded frame 28 from 28.png shape=(1000, 1000, 3)
Loaded frame 32 from 32.png shape=(1000, 1000, 3)
Loaded frame 37 from 37.png shape=(1000, 1000, 3)

# **Metrics Calculations(Done For Report)**

In [19]:
import cv2
import numpy as np
from pathlib import Path

CorrDir = Path("frames_corrupted")
RestDir = Path("frames_restored")
PairDir = Path("Metrics_Pairs")
PairDir.mkdir(parents=True, exist_ok=True)

FrameIdx = [0, 4, 9, 14, 18, 23, 28, 32, 37, 42]

def Mse(a: np.ndarray, b: np.ndarray) -> float:
    h = min(a.shape[0], b.shape[0]); w = min(a.shape[1], b.shape[1])
    A = a[:h, :w].astype(np.float32); B = b[:h, :w].astype(np.float32)
    return float(np.mean((A - B) ** 2))

def SsimSingleChannel(x: np.ndarray, y: np.ndarray, L: float = 255.0) -> float:
    K1, K2 = 0.01, 0.03
    C1, C2 = (K1 * L) ** 2, (K2 * L) ** 2
    mu_x = cv2.GaussianBlur(x, (11, 11), 1.5)
    mu_y = cv2.GaussianBlur(y, (11, 11), 1.5)
    mu_x2, mu_y2 = mu_x * mu_x, mu_y * mu_y
    mu_xy = mu_x * mu_y
    sigma_x2 = cv2.GaussianBlur(x * x, (11, 11), 1.5) - mu_x2
    sigma_y2 = cv2.GaussianBlur(y * y, (11, 11), 1.5) - mu_y2
    sigma_xy = cv2.GaussianBlur(x * y, (11, 11), 1.5) - mu_xy
    num = (2 * mu_xy + C1) * (2 * sigma_xy + C2)
    den = (mu_x2 + mu_y2 + C1) * (sigma_x2 + sigma_y2 + C2)
    return float((num / (den + 1e-12)).mean())

def Ssim(img1: np.ndarray, img2: np.ndarray) -> float:
    h = min(img1.shape[0], img2.shape[0]); w = min(img1.shape[1], img2.shape[1])
    A = img1[:h, :w]; B = img2[:h, :w]
    if A.ndim == 2:
        return SsimSingleChannel(A.astype(np.float32), B.astype(np.float32))
    if A.ndim == 3 and A.shape[2] in (3, 4):
        if A.shape[2] == 4: A = A[:, :, :3]
        if B.shape[2] == 4: B = B[:, :, :3]
        ch = [SsimSingleChannel(A[:, :, c].astype(np.float32), B[:, :, c].astype(np.float32)) for c in range(3)]
        return float(np.mean(ch))
    raise ValueError("Unsupported image shape for SSIM")

def ReadBgr(path: Path) -> np.ndarray:
    im = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if im is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return im

def PutTextOutlined(img, text, org, font_scale=1.4, fg=(255, 255, 255), bg=(0, 0, 0)):
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(img, text, org, font, font_scale, bg, 3, cv2.LINE_AA)
    cv2.putText(img, text, org, font, font_scale, fg, 1, cv2.LINE_AA)

def SavePair(idx: int, corrupted: np.ndarray, restored: np.ndarray, mse_val: float, ssim_val: float, out_path: Path):
    h = min(corrupted.shape[0], restored.shape[0])
    w = min(corrupted.shape[1], restored.shape[1])
    C = corrupted[:h, :w]
    R = restored[:h, :w]
    spacer = np.full((h, 6, 3), 255, dtype=np.uint8)
    pair = np.concatenate([C, spacer, R], axis=1)

    PutTextOutlined(pair, f"Corrupted {idx:03d}.png", (10, 28))
    PutTextOutlined(pair, f"Restored {idx}.png", (w + 16, 28))

    metrics_text = f"MSE: {mse_val:.2f}   SSIM: {ssim_val:.4f}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), _ = cv2.getTextSize(metrics_text, font, 1.4, 1)
    x_center = (pair.shape[1] - tw) // 2
    y_baseline = h - 12
    PutTextOutlined(pair, metrics_text, (x_center, y_baseline))

    cv2.imwrite(str(out_path), pair)

for i in FrameIdx:
    corr = ReadBgr(CorrDir / f"{i:03d}.png")
    rest = ReadBgr(RestDir / f"{i}.png")
    m = Mse(rest, corr)
    s = Ssim(rest, corr)
    out_path = PairDir / f"frame_{i:02d}_pair.png"
    SavePair(i, corr, rest, m, s, out_path)
    print(f"[{i:02d}] MSE={m:.2f}  SSIM={s:.4f}  -> {out_path.name}")

[00] MSE=5.04  SSIM=0.9797  -> frame_00_pair.png
[04] MSE=19409.45  SSIM=0.0905  -> frame_04_pair.png
[09] MSE=113.16  SSIM=0.8629  -> frame_09_pair.png
[14] MSE=84.01  SSIM=0.7855  -> frame_14_pair.png
[18] MSE=346.21  SSIM=0.5703  -> frame_18_pair.png
[23] MSE=958.85  SSIM=0.6490  -> frame_23_pair.png
[28] MSE=1035.21  SSIM=0.5870  -> frame_28_pair.png
[32] MSE=1389.80  SSIM=0.4432  -> frame_32_pair.png
[37] MSE=17707.89  SSIM=0.0854  -> frame_37_pair.png
[42] MSE=1145.92  SSIM=0.5133  -> frame_42_pair.png
